In [ ]:
# Cell 1: Imports & paths
import os, json, random, warnings, glob
import numpy as np
import torch
import torch.nn as nn
import librosa
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import brentq
from scipy.interpolate import interp1d
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAMPLE_RATE = 16000
print(f'Device: {DEVICE}')

# ── Auto-find data dirs (same approach as other phases) ───────────────────────
def _find_dirs():
    for root in ['/kaggle/input', '/kaggle/working']:
        for g in glob.glob(f'{root}/**/genuine', recursive=True):
            g = Path(g)
            if (g.parent / 'impostor').is_dir() and (g.parent / 'spoof').is_dir():
                return str(g), str(g.parent / 'impostor'), str(g.parent / 'spoof')
    return None, None, None

GENUINE_DIR, IMPOSTOR_DIR, SPOOF_DIR = _find_dirs()
if GENUINE_DIR is None:
    raise FileNotFoundError(
        'Could not find genuine/impostor/spoof dirs.\n'
        'Make sure the Phase 1 dataset is attached via + Add Data.'
    )
print(f'Genuine  : {GENUINE_DIR}')
print(f'Impostor : {IMPOSTOR_DIR}')
print(f'Spoof    : {SPOOF_DIR}')

# ── Auto-find model checkpoints ───────────────────────────────────────────────
def _find_ckpt(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True) + \
           glob.glob(f'/kaggle/working/**/{name}', recursive=True)
    return hits[0] if hits else None

PHASE2_CKPT = _find_ckpt('tdnn_best.pt')
PHASE4_CKPT = _find_ckpt('antispoof_best.pt')
print(f'Phase 2 ckpt: {PHASE2_CKPT}')
print(f'Phase 4 ckpt: {PHASE4_CKPT}')

RESULTS_DIR = '/kaggle/working/phase5/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# Cell 2: Rebuild model architectures — must match Phase 2 & Phase 4 EXACTLY
import torch.nn.functional as F

# ── Phase 2: TDNN (stats pooling, matches phase2_text_dependent.ipynb) ────────
class _TDNNLayer(nn.Module):
    def __init__(self, in_d, out_d, ctx=5, dil=1):
        super().__init__()
        self.conv = nn.Conv1d(in_d, out_d, kernel_size=ctx, dilation=dil,
                              padding=(ctx-1)*dil//2)
        self.bn = nn.BatchNorm1d(out_d)
    def forward(self, x): return F.relu(self.bn(self.conv(x)))

class TDNNSpeakerNet(nn.Module):
    def __init__(self, in_d=40, emb_d=256):
        super().__init__()
        self.tdnn = nn.Sequential(
            _TDNNLayer(in_d, 512, 5, 1), _TDNNLayer(512, 512, 3, 2),
            _TDNNLayer(512, 512, 3, 3),  _TDNNLayer(512, 512, 1, 1),
            _TDNNLayer(512, 1500, 1, 1),
        )
        self.fc1 = nn.Sequential(nn.Linear(3000, 512), nn.BatchNorm1d(512),
                                  nn.ReLU(), nn.Dropout(0.2))
        self.fc2 = nn.Sequential(nn.Linear(512, emb_d), nn.BatchNorm1d(emb_d))
    def forward(self, x):          # x: (B, T, D)
        x = self.tdnn(x.transpose(1, 2))          # → (B, 1500, T)
        x = torch.cat([x.mean(2), x.std(2)], 1)  # stats pool → (B, 3000)
        return F.normalize(self.fc2(self.fc1(x)), dim=1)

# ── Phase 4: AntiSpoofNet (input_dim=120, matches phase4_anti_spoofing.ipynb) ─
class FMS(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.scale = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Linear(channels, channels), nn.Sigmoid()
        )
    def forward(self, x): return x * self.scale(x).unsqueeze(-1)

class ResBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.fms   = FMS(out_ch)
        self.skip  = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, 1, stride=stride, bias=False),
            nn.BatchNorm1d(out_ch)
        ) if (in_ch != out_ch or stride != 1) else nn.Identity()
        self.drop  = nn.Dropout(0.1)
    def forward(self, x):
        h = F.leaky_relu(self.bn1(self.conv1(x)), 0.1)
        h = self.drop(self.bn2(self.conv2(h)))
        h = self.fms(h)
        return F.leaky_relu(h + self.skip(x), 0.1)

class AntiSpoofNet(nn.Module):
    def __init__(self, input_dim=120):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(input_dim, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm1d(64), nn.LeakyReLU(0.1)
        )
        self.layer1 = ResBlock1D(64,  128, stride=2)
        self.layer2 = ResBlock1D(128, 128)
        self.layer3 = ResBlock1D(128, 256, stride=2)
        self.layer4 = ResBlock1D(256, 256)
        self.classifier = nn.Sequential(
            nn.Linear(512, 256), nn.BatchNorm1d(256),
            nn.LeakyReLU(0.1), nn.Dropout(0.3),
            nn.Linear(256, 1)
        )
    def forward(self, x):
        x = x.permute(0, 2, 1)   # (B,T,F) → (B,F,T)
        h = self.stem(x)
        h = self.layer1(h); h = self.layer2(h)
        h = self.layer3(h); h = self.layer4(h)
        h = torch.cat([h.mean(-1), h.std(-1) + 1e-8], dim=-1)
        return self.classifier(h).squeeze(-1)

print('Model architectures defined.')


In [ ]:
# Cell 3: Feature extraction helpers
N_MFCC  = 40
MAX_LEN = 300

def load_wav(path, sr=SAMPLE_RATE):
    wav, _ = librosa.load(path, sr=sr, mono=True)
    return wav

def mfcc_p2(wav):
    """Phase 2 features: MFCC-40, shape (T, 40) — matches phase2 extract_mfcc."""
    feat = librosa.feature.mfcc(y=wav, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
                                  n_fft=512, hop_length=160, win_length=400)
    feat = (feat - feat.mean(1, keepdims=True)) / (feat.std(1, keepdims=True) + 1e-8)
    T = feat.shape[1]
    feat = feat[:, :MAX_LEN] if T >= MAX_LEN else np.pad(feat, ((0,0),(0,MAX_LEN-T)))
    return feat.T.astype(np.float32)   # (T, 40)

def mfcc_p4(wav):
    """Phase 4 features: MFCC+delta+delta2 = 120-dim, shape (T, 120)."""
    target = MAX_LEN * 160
    wav = np.pad(wav, (0, max(0, target - len(wav))))[:target]
    mfcc   = librosa.feature.mfcc(y=wav, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
                                    n_fft=512, hop_length=160, n_mels=80)
    delta  = librosa.feature.delta(mfcc, order=1)
    delta2 = librosa.feature.delta(mfcc, order=2)
    feat   = np.concatenate([mfcc, delta, delta2], axis=0)  # (120, T)
    feat   = (feat - feat.mean(1, keepdims=True)) / (feat.std(1, keepdims=True) + 1e-8)
    T = feat.shape[1]
    feat = feat[:, :MAX_LEN] if T >= MAX_LEN else np.pad(feat, ((0,0),(0,MAX_LEN-T)))
    return feat.T.astype(np.float32)   # (T, 120)

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

print('Feature helpers ready.')


In [ ]:
# Cell 4: Load Phase 2 TDNN + build enrollment centroid
print('Loading Phase 2 (TDNN) model...')

all_genuine = sorted([
    str(p) for p in Path(GENUINE_DIR).rglob('*.wav')
    if 'aug_' not in p.name
])
print(f'Genuine originals: {len(all_genuine)}')

tdnn_model = None
p2_ok      = False

if PHASE2_CKPT and os.path.exists(PHASE2_CKPT):
    state = torch.load(PHASE2_CKPT, map_location=DEVICE)
    # Phase 2 saves state_dict directly (not wrapped in a dict)
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']
    tdnn_model = TDNNSpeakerNet(in_d=40, emb_d=256).to(DEVICE)
    tdnn_model.load_state_dict(state)
    tdnn_model.eval()
    p2_ok = True
    print(f'Phase 2 loaded from {PHASE2_CKPT}')
else:
    print(f'WARNING: Phase 2 checkpoint not found: {PHASE2_CKPT}')
    print('Phase 2 score will be 0.5 — fusion uses Phase 3 + Phase 4 only.')

@torch.no_grad()
def get_tdnn_embedding(wav_path):
    feat = mfcc_p2(load_wav(wav_path))                          # (T, 40)
    t    = torch.FloatTensor(feat).unsqueeze(0).to(DEVICE)     # (1, T, 40)
    return tdnn_model(t).cpu().numpy().squeeze()

if p2_ok:
    td_orig = [f for f in all_genuine if Path(f).stem.startswith('td_')]
    if not td_orig:
        td_orig = all_genuine
    print(f'Enrollment files: {len(td_orig)}')
    enrollment_centroid_p2 = np.array([get_tdnn_embedding(f) for f in td_orig]).mean(axis=0)
    print(f'P2 centroid shape: {enrollment_centroid_p2.shape}')
else:
    enrollment_centroid_p2 = None

genuine_orig = all_genuine  # used in scoring cell


In [ ]:
# Cell 5: Load Phase 3 ECAPA-TDNN model + build enrollment centroid
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'speechbrain'], check=True)
print('speechbrain installed.')

from speechbrain.inference.speaker import SpeakerRecognition


ecapa_model = SpeakerRecognition.from_hparams(
    source='speechbrain/spkrec-ecapa-voxceleb',
    savedir='/kaggle/working/pretrained_ecapa'
)

@torch.no_grad()
def get_ecapa_embedding(wav_path):
    import torchaudio
    wav, sr = torchaudio.load(wav_path)
    if sr != 16000:
        wav = torchaudio.functional.resample(wav, sr, 16000)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    emb = ecapa_model.encode_batch(wav)
    return emb.squeeze().cpu().numpy()

# Enrollment centroid from all genuine originals (ti_ and td_)
genuine_orig = [f for f in all_genuine]  # already filtered to no aug_ above
print(f'Building ECAPA enrollment centroid from {len(genuine_orig)} files...')
ecapa_embeds = np.array([get_ecapa_embedding(f) for f in genuine_orig])
enrollment_centroid_p3 = ecapa_embeds.mean(axis=0)
print(f'Phase 3 enrollment centroid shape: {enrollment_centroid_p3.shape}')

In [ ]:
# Cell 6: Load Phase 4 AntiSpoofNet
print('Loading Phase 4 (AntiSpoofNet) model...')

spoof_model = None
p4_ok       = False

if PHASE4_CKPT and os.path.exists(PHASE4_CKPT):
    state = torch.load(PHASE4_CKPT, map_location=DEVICE)
    # Phase 4 saves state_dict directly (not wrapped in a dict)
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']
    spoof_model = AntiSpoofNet(input_dim=120).to(DEVICE)
    spoof_model.load_state_dict(state)
    spoof_model.eval()
    p4_ok = True
    print(f'Phase 4 loaded from {PHASE4_CKPT}')
else:
    print(f'WARNING: Phase 4 checkpoint not found: {PHASE4_CKPT}')
    print('Phase 4 score will be 0.5 — fusion uses Phase 2 + Phase 3 only.')

@torch.no_grad()
def get_spoof_score(wav_path):
    """Returns P(bonafide) in [0,1]. Higher = more likely genuine.""".
    if not p4_ok:
        return 0.5
    feat  = mfcc_p4(load_wav(wav_path))                        # (T, 120)
    t     = torch.FloatTensor(feat).unsqueeze(0).to(DEVICE)   # (1, T, 120)
    logit = spoof_model(t)
    return float(torch.sigmoid(logit).cpu().item())

print('Phase 4 ready.')


In [ ]:
# Cell 7: Score all test files
import os, glob
from pathlib import Path

# Re-discover paths if not already defined (self-contained)
if 'IMPOSTOR_DIR' not in dir() or IMPOSTOR_DIR is None:
    def _find_dirs():
        for root in ['/kaggle/input', '/kaggle/working']:
            for g in glob.glob(f'{root}/**/genuine', recursive=True):
                g = Path(g)
                if (g.parent / 'impostor').is_dir() and (g.parent / 'spoof').is_dir():
                    return str(g), str(g.parent / 'impostor'), str(g.parent / 'spoof')
        return None, None, None
    GENUINE_DIR, IMPOSTOR_DIR, SPOOF_DIR = _find_dirs()
    if IMPOSTOR_DIR is None:
        raise FileNotFoundError('Could not find impostor/spoof dirs. Attach Phase 1 dataset.')

if 'genuine_orig' not in dir() or genuine_orig is None:
    genuine_orig = sorted([
        str(p) for p in Path(GENUINE_DIR).rglob('*.wav')
        if 'aug_' not in p.name
    ])

# genuine originals = target speaker
# impostor files    = different speakers (LibriSpeech)
# spoof files       = synthetic / TTS
impostor_files = sorted([str(p) for p in Path(IMPOSTOR_DIR).rglob('*.wav')])
spoof_files    = sorted([str(p) for p in Path(SPOOF_DIR).rglob('*.wav')])

test_files  = genuine_orig + impostor_files + spoof_files
true_labels = ([1] * len(genuine_orig) +
               [0] * len(impostor_files) +
               [0] * len(spoof_files))
file_types  = (['genuine'] * len(genuine_orig) +
               ['impostor'] * len(impostor_files) +
               ['spoof'] * len(spoof_files))

print(f'Test set: {len(genuine_orig)} genuine | {len(impostor_files)} impostor | {len(spoof_files)} spoof')
print('Extracting scores... (this may take a few minutes)')

scores_p2, scores_p3, scores_p4 = [], [], []
for i, fpath in enumerate(test_files):
    if p2_ok:
        emb2 = get_tdnn_embedding(fpath)
        s2 = cosine_sim(emb2, enrollment_centroid_p2)
        s2 = (s2 + 1) / 2
    else:
        s2 = 0.5
    scores_p2.append(s2)

    emb3 = get_ecapa_embedding(fpath)
    s3 = cosine_sim(emb3, enrollment_centroid_p3)
    s3 = (s3 + 1) / 2
    scores_p3.append(s3)

    s4 = get_spoof_score(fpath)
    scores_p4.append(s4)

    if (i + 1) % 50 == 0:
        print(f'  [{i+1}/{len(test_files)}]')

import numpy as np
scores_p2 = np.array(scores_p2)
scores_p3 = np.array(scores_p3)
scores_p4 = np.array(scores_p4)
y_true    = np.array(true_labels)
print('Scoring complete.')

In [ ]:
# Cell 8: Metric helpers
def compute_eer(y_true, scores):
    fpr, tpr, thr = roc_curve(y_true, scores)
    eer = brentq(lambda x: 1.0 - x - interp1d(fpr, tpr)(x), 0.0, 1.0)
    thr_eer = interp1d(fpr, thr)(eer)
    return float(eer), float(thr_eer), fpr, tpr, thr

def tar_at_far(fpr, tpr, far_target):
    idx = np.searchsorted(fpr, far_target)
    idx = min(idx, len(tpr) - 1)
    return float(tpr[idx])

def compute_min_tdcf(fpr, tpr, thr,
                     p_spoof=0.05, c_miss=1.0, c_fa=10.0):
    fnr = 1 - tpr
    dcf  = c_miss * fnr * (1 - p_spoof) + c_fa * fpr * p_spoof
    return float(dcf.min())

def evaluate(y_true, scores, label=''):
    auc  = roc_auc_score(y_true, scores)
    eer, thr, fpr, tpr, thrs = compute_eer(y_true, scores)
    tar1  = tar_at_far(fpr, tpr, 0.01)
    tar01 = tar_at_far(fpr, tpr, 0.001)
    tdcf  = compute_min_tdcf(fpr, tpr, thrs)
    print(f'{label}:')
    print(f'  EER={eer*100:.2f}%  AUC={auc*100:.2f}%  TAR@FAR1%={tar1*100:.2f}%'
          f'  TAR@FAR0.1%={tar01*100:.2f}%  min-tDCF={tdcf:.4f}')
    return dict(eer=eer, auc=auc, tar1=tar1, tar01=tar01, tdcf=tdcf,
                fpr=fpr.tolist(), tpr=tpr.tolist())

print('Metric helpers ready.')

In [ ]:
# Cell 9: Individual system scores
print('=' * 60)
r2 = evaluate(y_true, scores_p2, 'Phase 2 (TDNN text-dependent)')
r3 = evaluate(y_true, scores_p3, 'Phase 3 (ECAPA-TDNN text-indep.)')
r4 = evaluate(y_true, scores_p4, 'Phase 4 (AntiSpoofNet)')
print('=' * 60)

In [ ]:
# Cell 10: Score fusion
# ── Strategy A: Weighted Sum ─────────────────────────────────────────────────
# Heuristic weights; higher weight to the subsystem with lower individual EER
eer_list = [r2['eer'], r3['eer'], r4['eer']]
inv_eers = [1.0 / (e + 1e-9) for e in eer_list]
total    = sum(inv_eers)
w2, w3, w4 = [v / total for v in inv_eers]
print(f'Auto weights: P2={w2:.3f}  P3={w3:.3f}  P4={w4:.3f}')

scores_fusion_weighted = w2 * scores_p2 + w3 * scores_p3 + w4 * scores_p4
print('\n--- Weighted Sum Fusion ---')
r_ws = evaluate(y_true, scores_fusion_weighted, 'Weighted Sum')

# ── Strategy B: Equal-Weight Sum ─────────────────────────────────────────────
scores_fusion_equal = (scores_p2 + scores_p3 + scores_p4) / 3.0
print('\n--- Equal-Weight Sum Fusion ---')
r_eq = evaluate(y_true, scores_fusion_equal, 'Equal Sum')

# ── Strategy C: Product Rule ──────────────────────────────────────────────────
# Assumes conditional independence; scores must be probabilities in (0,1)
eps = 1e-6
scores_fusion_product = (np.clip(scores_p2, eps, 1-eps) *
                          np.clip(scores_p3, eps, 1-eps) *
                          np.clip(scores_p4, eps, 1-eps))
print('\n--- Product Rule Fusion ---')
r_pr = evaluate(y_true, scores_fusion_product, 'Product Rule')

# ── Strategy D: Logistic Regression Meta-Classifier ──────────────────────────
X_meta = np.column_stack([scores_p2, scores_p3, scores_p4])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
lr_preds = np.zeros(len(y_true))
for tr_idx, te_idx in skf.split(X_meta, y_true):
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_meta[tr_idx])
    X_te = scaler.transform(X_meta[te_idx])
    clf  = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
    clf.fit(X_tr, y_true[tr_idx])
    lr_preds[te_idx] = clf.predict_proba(X_te)[:, 1]

print('\n--- Logistic Regression Meta-Classifier (5-fold) ---')
r_lr = evaluate(y_true, lr_preds, 'LR Meta-Classifier')
print('=' * 60)

In [ ]:
# Cell 11: Score normalization (z-norm) before fusion
def znorm(s):
    return (s - s.mean()) / (s.std() + 1e-9)

s2_z = znorm(scores_p2)
s3_z = znorm(scores_p3)
s4_z = znorm(scores_p4)

# Map z-normed scores back to [0,1] via sigmoid for comparability
def sigmoid(x): return 1 / (1 + np.exp(-x))
scores_znorm_fusion = w2 * sigmoid(s2_z) + w3 * sigmoid(s3_z) + w4 * sigmoid(s4_z)
print('\n--- Z-Norm + Weighted Sum ---')
r_zn = evaluate(y_true, scores_znorm_fusion, 'Z-Norm Weighted Sum')

# Pick best strategy by EER
strategies = {
    'Weighted Sum':       (scores_fusion_weighted, r_ws),
    'Equal Sum':          (scores_fusion_equal,    r_eq),
    'Product Rule':       (scores_fusion_product,  r_pr),
    'LR Meta-Classifier': (lr_preds,               r_lr),
    'Z-Norm Weighted':    (scores_znorm_fusion,     r_zn),
}
best_name = min(strategies, key=lambda k: strategies[k][1]['eer'])
best_scores, best_res = strategies[best_name]
print(f'\nBest strategy: {best_name}  (EER={best_res["eer"]*100:.2f}%)')

In [ ]:
# Cell 12: Comprehensive visualization (3×3 grid)
fig = plt.figure(figsize=(18, 16))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

COLORS = {'P2': '#4C72B0', 'P3': '#DD8452', 'P4': '#55A868',
          'WS': '#C44E52', 'EQ': '#8172B3', 'PR': '#937860',
          'LR': '#DA8BC3', 'ZN': '#8C8C8C'}

def plot_roc(ax, fpr, tpr, label, color, eer):
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{label} (EER={eer*100:.1f}%)')

# ── (0,0) ROC: individual systems ─────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 0])
for sys, res, col in [
    ('Phase 2 TDNN',   r2, COLORS['P2']),
    ('Phase 3 ECAPA',  r3, COLORS['P3']),
    ('Phase 4 AntiSpoof', r4, COLORS['P4']),
]:
    ax.plot(res['fpr'], res['tpr'], color=col, lw=2, label=f'{sys} (EER={res["eer"]*100:.1f}%)')
ax.plot([0,1],[0,1],'k--',alpha=0.4); ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('Individual System ROC'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# ── (0,1) ROC: fusion strategies ──────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 1])
fus_list = [
    ('Weighted Sum', r_ws, COLORS['WS']),
    ('Equal Sum',    r_eq, COLORS['EQ']),
    ('Product Rule', r_pr, COLORS['PR']),
    ('LR Meta-CL',   r_lr, COLORS['LR']),
    ('Z-Norm WS',    r_zn, COLORS['ZN']),
]
for name, res, col in fus_list:
    ax.plot(res['fpr'], res['tpr'], color=col, lw=2, label=f'{name} (EER={res["eer"]*100:.1f}%)')
ax.plot([0,1],[0,1],'k--',alpha=0.4); ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('Fusion Strategy ROC'); ax.legend(fontsize=7); ax.grid(alpha=0.3)

# ── (0,2) EER comparison bar chart ────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 2])
all_names = ['P2 TDNN', 'P3 ECAPA', 'P4 Anti', 'WSum', 'ESum', 'Prod', 'LR', 'ZNorm']
all_eers  = [r2['eer'], r3['eer'], r4['eer'],
             r_ws['eer'], r_eq['eer'], r_pr['eer'], r_lr['eer'], r_zn['eer']]
bar_cols  = [COLORS['P2'], COLORS['P3'], COLORS['P4'],
             COLORS['WS'], COLORS['EQ'], COLORS['PR'], COLORS['LR'], COLORS['ZN']]
bars = ax.bar(range(len(all_names)), [e*100 for e in all_eers], color=bar_cols)
ax.set_xticks(range(len(all_names))); ax.set_xticklabels(all_names, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('EER (%)'); ax.set_title('EER Comparison'); ax.grid(axis='y', alpha=0.3)
for bar, v in zip(bars, all_eers):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{v*100:.1f}%', ha='center', va='bottom', fontsize=7)

# ── (1,0) Score distribution: genuine vs impostor vs spoof (best fusion) ──────
ax = fig.add_subplot(gs[1, 0])
n_g = len(genuine_orig); n_i = len(impostor_files); n_s = len(spoof_files)
ax.hist(best_scores[:n_g],          bins=40, alpha=0.6, color='green',  label='Genuine')
ax.hist(best_scores[n_g:n_g+n_i],   bins=40, alpha=0.6, color='orange', label='Impostor')
ax.hist(best_scores[n_g+n_i:],      bins=40, alpha=0.6, color='red',    label='Spoof')
_, thr_best, *_ = compute_eer(y_true, best_scores)
ax.axvline(thr_best, color='black', ls='--', label=f'EER thr={thr_best:.3f}')
ax.set_xlabel('Fused Score'); ax.set_ylabel('Count')
ax.set_title(f'Score Distribution\n({best_name})')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# ── (1,1) Score distribution per subsystem (box plots) ────────────────────────
ax = fig.add_subplot(gs[1, 1])
groups = ['Genuine', 'Impostor', 'Spoof']
slices = [slice(0, n_g), slice(n_g, n_g+n_i), slice(n_g+n_i, None)]
data_box = []
for sys_scores, sys_label in [
    (scores_p2, 'P2'), (scores_p3, 'P3'), (scores_p4, 'P4'), (best_scores, 'Fused')
]:
    for sl, grp in zip(slices, groups):
        data_box.append({'scores': sys_scores[sl], 'sys': sys_label, 'grp': grp})

positions = []; labels_box = []; all_data = []
pos = 1
for grp in groups:
    for sys_label in ['P2', 'P3', 'P4', 'Fused']:
        entry = next(d for d in data_box if d['sys']==sys_label and d['grp']==grp)
        all_data.append(entry['scores'])
        positions.append(pos); labels_box.append(sys_label)
        pos += 1
    pos += 1
bp = ax.boxplot(all_data, positions=positions, widths=0.7,
                patch_artist=True, showfliers=False)
sys_colors = {'P2': COLORS['P2'], 'P3': COLORS['P3'], 'P4': COLORS['P4'], 'Fused': COLORS['WS']}
for patch, lbl in zip(bp['boxes'], labels_box):
    patch.set_facecolor(sys_colors[lbl]); patch.set_alpha(0.7)
ax.set_xticks([2.5, 7.5, 12.5]); ax.set_xticklabels(groups)
ax.set_ylabel('Score'); ax.set_title('Score Box-Plots by Group')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=c, label=l) for l, c in sys_colors.items()], fontsize=8)
ax.grid(axis='y', alpha=0.3)

# ── (1,2) DET curves ──────────────────────────────────────────────────────────
from scipy.stats import norm as scipy_norm
ax = fig.add_subplot(gs[1, 2])
for name, res, col in [('P2 TDNN', r2, COLORS['P2']),
                        ('P3 ECAPA', r3, COLORS['P3']),
                        ('P4 Anti', r4, COLORS['P4']),
                        (best_name[:8], best_res, COLORS['WS'])]:
    fpr_d = np.array(res['fpr']); tpr_d = np.array(res['tpr'])
    fnr_d = 1 - tpr_d
    mask = (fpr_d > 0) & (fnr_d > 0)
    ax.plot(scipy_norm.ppf(fpr_d[mask]), scipy_norm.ppf(fnr_d[mask]),
            color=col, lw=2, label=name)
ax.set_xlabel('FPR (probit)'); ax.set_ylabel('FNR (probit)')
ax.set_title('DET Curves'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# ── (2,0) min-tDCF bar chart ──────────────────────────────────────────────────
ax = fig.add_subplot(gs[2, 0])
all_tdcf = [r2['tdcf'], r3['tdcf'], r4['tdcf'],
             r_ws['tdcf'], r_eq['tdcf'], r_pr['tdcf'], r_lr['tdcf'], r_zn['tdcf']]
bars = ax.bar(range(len(all_names)), all_tdcf, color=bar_cols)
ax.set_xticks(range(len(all_names))); ax.set_xticklabels(all_names, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('min-tDCF'); ax.set_title('min-tDCF Comparison'); ax.grid(axis='y', alpha=0.3)
for bar, v in zip(bars, all_tdcf):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
            f'{v:.4f}', ha='center', va='bottom', fontsize=7)

# ── (2,1) AUC bar chart ───────────────────────────────────────────────────────
ax = fig.add_subplot(gs[2, 1])
all_auc = [r2['auc'], r3['auc'], r4['auc'],
            r_ws['auc'], r_eq['auc'], r_pr['auc'], r_lr['auc'], r_zn['auc']]
bars = ax.bar(range(len(all_names)), [a*100 for a in all_auc], color=bar_cols)
ax.set_xticks(range(len(all_names))); ax.set_xticklabels(all_names, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('AUC (%)'); ax.set_title('AUC Comparison')
ax.set_ylim([max(0, min(all_auc)*100 - 2), 101])
ax.grid(axis='y', alpha=0.3)
for bar, v in zip(bars, all_auc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{v*100:.1f}%', ha='center', va='bottom', fontsize=7)

# ── (2,2) Summary text box ────────────────────────────────────────────────────
ax = fig.add_subplot(gs[2, 2])
ax.axis('off')
summary = [
    'SCORE FUSION SUMMARY',
    '=' * 30,
    f'Dataset: {n_g} genuine | {n_i} impostor | {n_s} spoof',
    '',
    'Individual Systems:',
    f'  P2 TDNN:   EER={r2["eer"]*100:.2f}% AUC={r2["auc"]*100:.2f}%',
    f'  P3 ECAPA:  EER={r3["eer"]*100:.2f}% AUC={r3["auc"]*100:.2f}%',
    f'  P4 AntiSpoof: EER={r4["eer"]*100:.2f}% AUC={r4["auc"]*100:.2f}%',
    '',
    'Fusion Strategies:',
    f'  Weighted Sum: EER={r_ws["eer"]*100:.2f}%',
    f'  Equal Sum:    EER={r_eq["eer"]*100:.2f}%',
    f'  Product Rule: EER={r_pr["eer"]*100:.2f}%',
    f'  LR Meta-CL:   EER={r_lr["eer"]*100:.2f}%',
    f'  Z-Norm WS:    EER={r_zn["eer"]*100:.2f}%',
    '',
    f'BEST: {best_name}',
    f'  EER  = {best_res["eer"]*100:.2f}%',
    f'  AUC  = {best_res["auc"]*100:.2f}%',
    f'  TAR@FAR1%  = {best_res["tar1"]*100:.2f}%',
    f'  TAR@FAR0.1% = {best_res["tar01"]*100:.2f}%',
    f'  min-tDCF = {best_res["tdcf"]:.4f}',
]
ax.text(0.05, 0.95, '\n'.join(summary), transform=ax.transAxes,
        va='top', ha='left', fontsize=8.5, family='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

fig.suptitle('Phase 5: Score Fusion — Complete Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.savefig(f'{RESULTS_DIR}/phase5_fusion.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

In [ ]:
# Cell 13: Save results + optimal threshold for Gradio UI
_, optimal_threshold, *_ = compute_eer(y_true, best_scores)

results = {
    'best_strategy': best_name,
    'optimal_threshold': float(optimal_threshold),
    'weights': {'p2': float(w2), 'p3': float(w3), 'p4': float(w4)},
    'individual': {
        'phase2_tdnn':      {k: float(v) for k, v in r2.items() if k not in ('fpr','tpr')},
        'phase3_ecapa':     {k: float(v) for k, v in r3.items() if k not in ('fpr','tpr')},
        'phase4_antispoof': {k: float(v) for k, v in r4.items() if k not in ('fpr','tpr')},
    },
    'fusion': {
        'weighted_sum':       {k: float(v) for k, v in r_ws.items() if k not in ('fpr','tpr')},
        'equal_sum':          {k: float(v) for k, v in r_eq.items() if k not in ('fpr','tpr')},
        'product_rule':       {k: float(v) for k, v in r_pr.items() if k not in ('fpr','tpr')},
        'lr_meta_classifier': {k: float(v) for k, v in r_lr.items() if k not in ('fpr','tpr')},
        'znorm_weighted':     {k: float(v) for k, v in r_zn.items() if k not in ('fpr','tpr')},
    },
    'best': {k: float(v) for k, v in best_res.items() if k not in ('fpr','tpr')},
}

out_path = f'{RESULTS_DIR}/phase5_summary.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved to {out_path}')

# Save enrollment centroids for Phase 6 (Gradio UI)
models_dir = '/kaggle/working/phase5/models'
os.makedirs(models_dir, exist_ok=True)
if enrollment_centroid_p2 is not None:
    np.save(f'{models_dir}/enrollment_centroid_p2.npy', enrollment_centroid_p2)
    print('P2 centroid saved.')
np.save(f'{models_dir}/enrollment_centroid_p3.npy', enrollment_centroid_p3)
print('P3 centroid saved.')

print(f'
=== PHASE 5 COMPLETE ===")
print(f'Best fusion: {best_name}')
print(f'EER  = {best_res["eer"]*100:.2f}%')
print(f'AUC  = {best_res["auc"]*100:.2f}%')
print(f'TAR@FAR1%   = {best_res["tar1"]*100:.2f}%')
print(f'TAR@FAR0.1% = {best_res["tar01"]*100:.2f}%')
print(f'min-tDCF    = {best_res["tdcf"]:.4f}')
print(f'Optimal threshold = {optimal_threshold:.4f}')
